# Inference Notebook

Loads saved OOF/test predictions from multiple experiments and blends them.

CPU only. No model retraining here.

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.metrics import roc_auc_score

# Config
DATA_DIR   = Path('../data/raw')
RESULT_DIR = Path('../results/experiments')
TARGET_COL = 'target'

BLEND_CONFIG = [
    {'exp_id': 'EXP-001', 'weight': 1.0},
    # {'exp_id': 'EXP-002', 'weight': 1.0},
]

print('Blend config:', BLEND_CONFIG)

In [ ]:
# Load ground truth for OOF scoring
train_path = DATA_DIR / 'train.csv'
if train_path.exists():
    train = pd.read_csv(train_path)
    y_true = train[TARGET_COL].values
else:
    y_true = None
    print('WARNING: train.csv not found, skipping OOF scoring')

In [ ]:
# Load and blend predictions
oof_blend    = None
test_blend   = None
total_weight = sum(c['weight'] for c in BLEND_CONFIG)

for cfg in BLEND_CONFIG:
    exp_id    = cfg['exp_id']
    w         = cfg['weight'] / total_weight
    oof_path  = RESULT_DIR / f'{exp_id}_oof.npy'
    test_path = RESULT_DIR / f'{exp_id}_test_pred.npy'

    if not oof_path.exists():
        print(f'MISSING: {oof_path} -- skipping {exp_id}')
        continue

    oof  = np.load(oof_path)
    test = np.load(test_path) if test_path.exists() else np.zeros(1)

    if oof_blend is None:
        oof_blend  = np.zeros_like(oof,  dtype=float)
        test_blend = np.zeros_like(test, dtype=float)

    oof_blend  += w * oof
    test_blend += w * test

    if y_true is not None and len(y_true) == len(oof):
        score = roc_auc_score(y_true, oof)
        print(f'{exp_id}  OOF AUC = {score:.5f}  weight = {w:.3f}')

if oof_blend is not None and y_true is not None and len(y_true) == len(oof_blend):
    blend_score = roc_auc_score(y_true, oof_blend)
    print(f'\nBlend OOF AUC = {blend_score:.5f}')
else:
    print('No blended predictions loaded.')

In [ ]:
# Save blended test predictions
if test_blend is not None:
    blend_ids = '_'.join(c['exp_id'] for c in BLEND_CONFIG)
    out_path  = RESULT_DIR / f'blend_{blend_ids}_test.npy'
    np.save(out_path, test_blend)
    print(f'Saved: {out_path}')
    print('Next: run 04_submission.ipynb')